# 支持修补对比

- 对应论文章节：第3.3.2节 生成侧补充对比实验
- 源脚本：`experiments/03_证伪实验/scripts/运行_支持修补对比.py`
- notebook 作用：直接查看代码与已保存结果，命令行运行仍以 `.py` 为准

这本 notebook 单独展示生成后支持性修补和基线、对齐作答之间的差异。

## 命令行复现

```bash
cd /root/Velo
/root/Velo/.venv/bin/python experiments/03_证伪实验/scripts/运行_支持修补对比.py
```

## 源码镜像

下面这一格保留 `.py` 的完整源码，主要用于现场查阅。

In [ ]:
"""比较 baseline、对齐作答与支持性修补三种生成路线。"""

from __future__ import annotations

import argparse
import json
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
EXPERIMENTS_ROOT = ROOT.parent
IMPL_ROOT = EXPERIMENTS_ROOT / "04_算法实现"
if str(IMPL_ROOT) not in sys.path:
    sys.path.insert(0, str(IMPL_ROOT))

from retrieval_pipeline.common import DEFAULT_EMBEDDING_MODEL, DEFAULT_LLM_MODEL, ensure_dir
from retrieval_pipeline.datasets import load_crud_cases
from retrieval_pipeline.metrics import evaluate_crud_results
from retrieval_pipeline.pipeline import PipelineVariant, RagExperimentPipeline

OUTPUT_ROOT = ROOT / "results" / "02_生成侧补充对比"

COMPLEX_CASE_IDS = (
    "questanswer_2docs_002",
    "questanswer_2docs_003",
    "questanswer_2docs_005",
    "questanswer_2docs_006",
    "questanswer_2docs_008",
    "questanswer_3docs_003",
    "questanswer_3docs_004",
    "questanswer_3docs_006",
)

VARIANTS = (
    PipelineVariant(
        key="baseline_simple",
        label="基线直接作答",
        use_rerank=True,
        answer_prompt_style="simple",
        multi_snippet_count=2,
    ),
    PipelineVariant(
        key="aligned_direct",
        label="对齐作答",
        use_rerank=True,
        answer_prompt_style="aligned",
        multi_snippet_count=2,
    ),
    PipelineVariant(
        key="support_repair",
        label="支持性修补",
        use_rerank=True,
        synthesis_mode="support_repair",
        answer_prompt_style="aligned",
        support_pruning_threshold=0.55,
        multi_snippet_count=2,
    ),
)


def run_variant_on_cases(pipeline: RagExperimentPipeline, prepared, variant: PipelineVariant):
    results = []
    total = len(prepared.cases)
    for index, case in enumerate(prepared.cases, start=1):
        if index == 1 or index == total:
            print(f"[support-repair] {variant.key}: {index}/{total}", flush=True)
        results.append(pipeline.run_case(prepared, case, variant))
    return results


def main() -> None:
    parser = argparse.ArgumentParser(description="运行支持性修补对比实验。")
    parser.add_argument("--llm-model", default=DEFAULT_LLM_MODEL)
    args = parser.parse_args()

    ensure_dir(OUTPUT_ROOT)
    cache_root = ensure_dir(ROOT / ".cache")

    cases, docs = load_crud_cases(
        summary_samples=6,
        qa_1doc_samples=6,
        qa_2doc_samples=8,
        qa_3doc_samples=8,
        hallu_samples=4,
        negative_samples=6,
        distractor_count=600,
        seed=42,
    )
    selected_cases = [case for case in cases if case.case_id in COMPLEX_CASE_IDS]
    if len(selected_cases) != len(COMPLEX_CASE_IDS):
        found = {case.case_id for case in selected_cases}
        missing = [case_id for case_id in COMPLEX_CASE_IDS if case_id not in found]
        raise RuntimeError(f"缺少复杂验证样例: {missing}")

    summaries = []
    details = []
    for variant in VARIANTS:
        pipeline = RagExperimentPipeline(
            cache_root=cache_root,
            embedding_model=DEFAULT_EMBEDDING_MODEL,
            llm_model=args.llm_model,
        )
        prepared = pipeline.prepare_dataset(
            "crud_support_repair_batch",
            selected_cases,
            docs,
            include_contextual=False,
            include_parent_child=False,
            include_query_rewrite=False,
        )
        results = run_variant_on_cases(pipeline, prepared, variant)
        evaluation = evaluate_crud_results(
            variant.key,
            results,
            selected_cases,
            ragas_case_ids=(),
            qa_ragas_case_ids=(),
            multidoc_ragas_case_ids=(),
            enable_ragas=False,
        )
        summary = dict(evaluation.summary)
        summary["label"] = variant.label
        summaries.append(summary)
        details.extend(evaluation.detail_rows)

    output = {
        "case_ids": list(COMPLEX_CASE_IDS),
        "llm_model": args.llm_model,
        "summaries": summaries,
    }
    (OUTPUT_ROOT / "生成侧_支持性修补对比_结果汇总.json").write_text(
        json.dumps(output, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    (OUTPUT_ROOT / "生成侧_支持性修补对比_逐题明细.json").write_text(
        json.dumps(details, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    print(json.dumps(output, ensure_ascii=False, indent=2))


if __name__ == "__main__":
    main()


## 结果预览

下面直接内嵌当前已保存结果的关键文件预览。

### 支持性修补对比结果

- 文件：`../results/02_生成侧补充对比/生成侧_支持性修补对比_结果汇总.json`

In [1]:
from pathlib import Path
import json

path = Path('../results/02_生成侧补充对比/生成侧_支持性修补对比_结果汇总.json')
data = json.loads(path.read_text(encoding='utf-8'))
if isinstance(data, list) and len(data) > 6:
    data = {'total_items': len(data), 'preview': data[:6]}
elif isinstance(data, dict):
    data = dict(data)
    for key in ('summaries', 'preview', 'rows'):
        value = data.get(key)
        if isinstance(value, list) and len(value) > 6:
            data[key] = {'total_items': len(value), 'preview': value[:6]}
print(json.dumps(data, ensure_ascii=False, indent=2))


{
  "case_ids": [
    "questanswer_2docs_002",
    "questanswer_2docs_003",
    "questanswer_2docs_005",
    "questanswer_2docs_006",
    "questanswer_2docs_008",
    "questanswer_3docs_003",
    "questanswer_3docs_004",
    "questanswer_3docs_006"
  ],
  "summaries": [
    {
      "variant": "baseline_simple",
      "dataset": "crud",
      "faithfulness": 0.0,
      "answer_correctness": 0.0,
      "answer_relevancy": 0.0,
      "context_precision": 0.0,
      "ragas_sample_count": 0,
      "accuracy": 0.0,
      "qa_accuracy": 0.0,
      "retrieval_hit_rate_at_1": 0.875,
      "retrieval_hit_rate_at_3": 1.0,
      "qa_faithfulness": 0.0,
      "qa_answer_correctness": 0.0,
      "qa_answer_relevancy": 0.0,
      "qa_context_precision": 0.0,
      "qa_ragas_sample_count": 0,
      "multidoc_faithfulness": 0.0,
      "multidoc_answer_correctness": 0.0,
      "multidoc_answer_relevancy": 0.0,
      "multidoc_context_precision": 0.0,
      "multidoc_ragas_sample_count": 0,
      "overal